# 04a — 贝叶斯优化（上）：高斯过程回归与预测不确定性量化

在第二讲（监督学习）和第三讲（无监督学习）中，我们处理的数据集通常包含上千甚至近万个分子（如 ESOL 的 1,128 个分子、AqSolDB 的 9,480 个分子）。但在真实的材料制备与器件工艺实验室里，情况往往恰恰相反：

- 合成一个新配体，或者旋涂、退火并表征一批钙钛矿/有机薄膜晶体管器件，往往需要耗费 **数天时间与昂贵的试剂**；
- 在课题起步阶段，你手头往往只有 **5 到 15 个实验数据点**；
- 面对尚未探索的庞大参数空间，你不仅需要模型给出一个预测值 $\hat{y}$，更迫切地需要知道**模型对这个预测有多大把握（不确定性 $\sigma(x)$ 有多大）**。

这正是**高斯过程回归（Gaussian Process Regression, GPR）**的核心价值：它不只拟合一条穿过数据点的确定性曲线，而是给出整条函数曲线的概率分布——在测过的实验点附近不确定性极小，而在尚未涉足的参数空白区，置信区间会自动张开，诚实地发出“此处缺乏数据”的预警。

本节我们将从二维相关正态分布出发，仅用 10 行 NumPy 代码手写实现高斯过程回归，再用 `scikit-learn` 处理含实验测量噪声的工艺数据，最后将连续空间的核函数推广到离散分子指纹空间的 **Tanimoto 核函数**。

---

## 0. 环境准备与数据读取辅助函数

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. 为什么材料实验迫切需要“预测不确定性”？

回顾第二讲 `02b`：当我们将训练好的随机森林模型直接用于预测并五苯（`pentacene`）的水溶解度时，模型毫不犹豫地输出了一个具体数值，却**没有任何内置机制提醒我们该分子已经远远超出了训练集的适用域**。

再考虑一个典型的材料工艺优化场景：假设我们在研究**有机半导体薄膜的退火温度 $T$（$60\ ^\circ\text{C} \sim 200\ ^\circ\text{C}$）与场效应载流子迁移率 $\mu$** 之间的关系。温度过低时分子结晶度不足，温度过高时薄膜容易开裂或发生相分离。如果我们先在实验室测了 6 个不同温度下的器件迁移率，中间还有一大段温度区间尚未测试：
- 普通回归模型（如多项式拟合或神经网络）只会画出一条独断的曲线；
- **高斯过程回归（GPR）**则会同时输出每个温度 $x$ 处的**预测均值 $\mu(x)$** 和**预测标准差 $\sigma(x)$**。在已测过的 6 个温度点附近，$\sigma(x)$ 收敛得很窄；而在未测过的温度空白区，$\sigma(x)$ 会显著鼓起。正是这种对未知区域的“自知之明”，构成了下一节（`04b`）贝叶斯优化自动设计下一轮实验的基石。

---

## 2. 从单变量正态分布到多维相关高斯分布

一维正态分布 $y \sim \mathcal{N}(\mu, \sigma^2)$ 描述单个随机变量的波动。如果我们在两个实验条件 $x_1$ 和 $x_2$ 下观测性质 $y_1, y_2$，可以将它们视为服从**二维联合正态分布**：

$$\begin{bmatrix} y_1 \\ y_2 \end{bmatrix} \sim \mathcal{N}\left(\begin{bmatrix} 0 \\ 0 \end{bmatrix}, \begin{bmatrix} 1 & \rho \\ \rho & 1 \end{bmatrix}\right)$$

非对角线元素 $\rho \in [-1, 1]$ 是两者的协方差（由于方差均为 1，也是相关系数）。**高斯过程的物理直觉非常朴素**：
- 如果两个实验条件 $x_1$ 和 $x_2$ 非常接近（例如退火温度只差 $2\ ^\circ\text{C}$），由于物理过程是连续光滑的，测出的性能 $y_1$ 和 $y_2$ 必然高度正相关（$\rho \to 1$）；
- 如果 $x_1$ 和 $x_2$ 相隔很远（例如退火温度相差 $100\ ^\circ\text{C}$），知道 $y_1$ 的数值对推测 $y_2$ 几乎没有帮助，两者的相关性趋近于零（$\rho \to 0$）。

下面从二维正态分布中随机采样 300 对 $(y_1, y_2)$，直观对比高相关性（$\rho = 0.9$）与弱相关性（$\rho = 0.1$）：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, rho in zip(axes, [0.9, 0.1]):
    cov = [[1.0, rho], [rho, 1.0]]
    samples = np.random.multivariate_normal([0, 0], cov, size=300)
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.45, s=18)
    ax.set_title(f"Correlation rho = {rho}")
    ax.set_xlabel("y1 at condition x1")
    ax.set_ylabel("y2 at condition x2")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. 核函数（Kernel）：从有限个点推广到连续随机函数

现在把上面的 2 个点推广到横轴上密密麻麻排列的 $N = 100$ 个网格点 $x_1, x_2, \dots, x_{100}$。此时向量 $\mathbf{y} = [f(x_1), \dots, f(x_{100})]^T$ 服从 100 维联合正态分布 $\mathcal{N}(\mathbf{0}, K)$，每抽样一个 100 维向量并连成折线，就等价于**随机画出了一条光滑函数曲线**。

那么，协方差矩阵 $K$ 的第 $(i, j)$ 个元素 $K_{ij} = \text{Cov}(f(x_i), f(x_j))$ 怎么由自变量距离 $|x_i - x_j|$ 决定？这就是**核函数（Kernel Function / Covariance Function）**的作用。

最经典的是 **RBF 核（径向基函数核 / 平方指数核，Radial Basis Function Kernel）**：

$$k(x_i, x_j) = \sigma_f^2 \exp\left(-\frac{\|x_i - x_j\|^2}{2\ell^2}\right)$$

其中包含两个具有明确物理意义的超参数：
1. **长度尺度 $\ell$（`length_scale`）**：决定函数沿横轴变化的平缓程度（即“相隔多远以内算作近邻”）。$\ell$ 越大，相距较远的点依然保持强相关，采样出的函数曲线越平缓；$\ell$ 越小，相关性随距离迅速衰减，函数曲线越曲折高频。
2. **信号振幅 $\sigma_f^2$（`constant_value`）**：控制函数值在纵轴方向上的整体波动幅度。

下面我们在尚未观测任何实验数据时，直接从 RBF 核定义的**高斯过程先验（GP Prior）**中随机抽取 5 条函数曲线：

In [ ]:
def rbf_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    """计算两组点集 X1 (n1, d) 与 X2 (n2, d) 之间的 RBF 协方差矩阵"""
    sq_dist = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * X1 @ X2.T
    return (sigma_f ** 2) * np.exp(-0.5 * sq_dist / (length_scale ** 2))

X_grid = np.linspace(0, 10, 100).reshape(-1, 1)

length_scale = 1.2   # 可尝试改为 0.3（剧烈震荡）或 3.0（极其平缓）观察变化
K_prior = rbf_kernel(X_grid, X_grid, length_scale=length_scale) + 1e-8 * np.eye(len(X_grid))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(K_prior, cmap="viridis", origin="lower")
axes[0].set_title(f"Prior Covariance Matrix K (length_scale = {length_scale})")
axes[0].set_xlabel("Grid Index j")
axes[0].set_ylabel("Grid Index i")
plt.colorbar(im, ax=axes[0], shrink=0.8)

prior_samples = np.random.multivariate_normal(np.zeros(len(X_grid)), K_prior, size=5)
for s in prior_samples:
    axes[1].plot(X_grid.ravel(), s, lw=1.8, alpha=0.85)
axes[1].set_title(f"5 Random Functions Drawn from GP Prior (l = {length_scale})")
axes[1].set_xlabel("x")
axes[1].set_ylabel("f(x)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. 仅用 10 行 NumPy 手写高斯过程回归（GP Posterior）

先验分布代表了模型在看到实验数据之前对所有光滑曲线的“合理猜测”。一旦我们在实验室测出了 $n$ 个数据点 $(X_{\text{train}}, \mathbf{y}_{\text{train}})$，所有不经过这些观测点（在观测噪声允许范围内）的先验曲线都被淘汰，留存下来的曲线束就构成了**高斯过程后验分布（GP Posterior）**。

由于训练观测值 $\mathbf{y}_{\text{train}}$ 与待预测位置 $X_*$ 上的函数值 $\mathbf{f}_*$ 服从联合高斯分布：

$$\begin{bmatrix} \mathbf{y}_{\text{train}} \\ \mathbf{f}_* \end{bmatrix} \sim \mathcal{N}\left(\mathbf{0}, \begin{bmatrix} K + \sigma_n^2 I & K_* \\ K_*^T & K_{**} \end{bmatrix}\right)$$

直接套用多元正态分布的条件概率公式，即可得到任意新位置 $X_*$ 上预测均值 $\boldsymbol{\mu}_*$ 与后验协方差矩阵 $\Sigma_*$ 的**精确解析解**：

$$\boldsymbol{\mu}_* = K_*^T (K + \sigma_n^2 I)^{-1} \mathbf{y}_{\text{train}}$$
$$\Sigma_* = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$$

其中：
- $K = k(X_{\text{train}}, X_{\text{train}})$ 是训练点之间的 $n \times n$ 核矩阵，$\sigma_n^2$ 是实验测量噪声方差；
- $K_* = k(X_{\text{train}}, X_*)$ 是训练点与待测网格点之间的 $n \times m$ 互协方差矩阵；
- $K_{**} = k(X_*, X_*)$ 是待测网格点自身的 $m \times m$ 先验协方差矩阵，后验协方差矩阵对角线元素的平方根 $\sigma_*(x) = \sqrt{\text{diag}(\Sigma_*)}$ 就是每个待测点上的**预测不确定性（标准差）**！

注意观察第二式 $\Sigma_* = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$ 的结构：**后验不确定性 = 先验不确定性 $K_{**}$ 减去因为观测到附近训练点而消减掉的不确定性**。如果某个待测点离所有已测训练点都很远，那么 $K_* \approx \mathbf{0}$，被减去的那一项消失，后验不确定性就会退回到先验不确定性（置信带重新张开）。

下面我们用 10 行 NumPy 代码实现这两个公式，拟合 6 个模拟的“退火温度 vs 载流子迁移率”实验点：

In [ ]:
# 模拟实验室已测的 6 个退火温度点（归一化到 0~10 区间，中间 3.5 ~ 7.2 为尚未测试的空白区）
X_train = np.array([1.0, 2.2, 3.5, 7.2, 8.5, 9.3]).reshape(-1, 1)

def true_mobility(x):
    return np.sin(0.9 * x) + 0.3 * np.cos(2.1 * x)

y_train = true_mobility(X_train).ravel()

def gp_posterior(X_tr, y_tr, X_te, length_scale=1.2, sigma_f=1.0, noise=1e-4):
    K    = rbf_kernel(X_tr, X_tr, length_scale, sigma_f) + noise * np.eye(len(X_tr))
    K_s  = rbf_kernel(X_tr, X_te, length_scale, sigma_f)
    K_ss = rbf_kernel(X_te, X_te, length_scale, sigma_f) + 1e-8 * np.eye(len(X_te))

    K_inv = np.linalg.inv(K)
    mu_s  = K_s.T @ K_inv @ y_tr
    cov_s = K_ss - K_s.T @ K_inv @ K_s
    return mu_s, cov_s

mu_s, cov_s = gp_posterior(X_train, y_train, X_grid, length_scale=1.2, noise=1e-4)
std_s = np.sqrt(np.maximum(np.diag(cov_s), 0))

plt.figure(figsize=(9, 4.5))
plt.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True Unknown Curve")
plt.plot(X_grid, mu_s, "b-", lw=2, label="GP Posterior Mean")
plt.fill_between(
    X_grid.ravel(), mu_s - 1.96 * std_s, mu_s + 1.96 * std_s,
    color="blue", alpha=0.18, label="95% Confidence Interval (±1.96 std)"
)

post_samples = np.random.multivariate_normal(mu_s, cov_s, size=3)
for i, s in enumerate(post_samples):
    plt.plot(X_grid.ravel(), s, lw=1, alpha=0.65, ls=":", label="Posterior Sample" if i == 0 else None)

plt.scatter(X_train, y_train, c="red", s=60, zorder=5, label="Lab Measurements (n = 6)")
plt.title("Hand-written Gaussian Process Regression on Annealing Temperature vs Mobility")
plt.xlabel("Normalized Annealing Temperature")
plt.ylabel("Normalized Mobility")
plt.legend(loc="lower left", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

观察上面的后验分布图：在 6 个红色实测点处，所有抽样曲线都收束穿过数据点，95% 置信区间收缩为 0；而在中间完全未做过实验的空白温度带 $x \in [3.5, 7.2]$ 以及两端边界外，蓝色阴影大幅膨胀，直观标示出了模型的信息盲区。

### 🏋️ 练习 04a-1：补充实验点如何压缩后验不确定性？

如果我们在空白区间和边界上补充测量 6 个新温度点，将训练集扩充为 12 个点：
`X_tr_12 = np.sort(np.vstack([X_train, np.array([[0.3], [4.2], [5.0], [5.8], [6.5], [9.8]])]), axis=0)`
1. 调用上面的 `gp_posterior` 分别计算 `n = 6` 和 `n = 12` 时在整个 `X_grid` 上的平均预测标准差 `std_s.mean()`；
2. 将 `n = 6` 和 `n = 12` 的预测均值与 95% 置信区间并排画在两张子图上，观察空白区不确定性的消减。

In [ ]:
# 在下方填写你的代码：


---

## 5. `scikit-learn` 实战：自动拟合核超参数与实验测量噪声

在手写代码中，我们人为指定了 `length_scale=1.2` 且假设测量几乎无噪声（`noise=1e-4`）。但在真实实验室中，任何材料表征都存在平行样涨落与仪器误差，而且我们事先并不知道最佳的 $\ell$ 和噪声水平 $\sigma_n^2$。

使用 `sklearn.gaussian_process.GaussianProcessRegressor` 时，可以通过组合算子构造复合核函数：
```python
kernel = C(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1)
```
- `C(1.0) * RBF(1.0)` 对应光滑信号项 $\sigma_f^2 \exp(-\|x - x'\|^2 / 2\ell^2)$；
- `WhiteKernel(noise_level=0.1)` 对应独立同分布的高斯白噪声项 $\sigma_n^2 \delta_{ij}$。
- 调用 `gp.fit(X_train, y_noisy)` 时，scikit-learn 会通过梯度优化**最大化边缘对数似然（Log-Marginal Likelihood）**，自动从数据中拟合出最合理的 $\sigma_f^2$、$\ell$ 和 $\sigma_n^2$（建议设置 `n_restarts_optimizer=10` 防止似然函数优化陷入局部极值）。

下面我们在 6 个实验点上加入标准差为 `0.15` 的随机测量误差，对比“不带 `WhiteKernel` 强行插值”与“带 `WhiteKernel` 自动识别噪声”的效果：

In [ ]:
rng = np.random.RandomState(7)
y_noisy = y_train + rng.normal(0, 0.15, size=len(y_train))

# 1. 不含噪声核（alpha 极小）：强行穿过每一个带噪数据点，导致长度尺度缩短、剧烈过拟合
gp_no_noise = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-6, n_restarts_optimizer=10, random_state=0)
gp_no_noise.fit(X_train, y_noisy)

# 2. 加上 WhiteKernel：允许模型区分底层光滑趋势与随机实验误差
kernel_noisy = C(1.0) * RBF(1.0) + WhiteKernel(noise_level=0.1)
gp_with_noise = GaussianProcessRegressor(kernel=kernel_noisy, n_restarts_optimizer=10, random_state=0)
gp_with_noise.fit(X_train, y_noisy)

print("不含 WhiteKernel 拟合出的核参数:", gp_no_noise.kernel_)
print("包含 WhiteKernel 拟合出的核参数:", gp_with_noise.kernel_)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, model, title in zip(
    axes,
    [gp_no_noise, gp_with_noise],
    ["Without WhiteKernel (Overfits Noise)", "With WhiteKernel (Smooth & Calibrated)"],
):
    mu, std = model.predict(X_grid, return_std=True)
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True Curve")
    ax.plot(X_grid, mu, "b-", lw=2, label="GP Mean")
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.18, label="95% CI")
    ax.scatter(X_train, y_noisy, c="red", s=55, zorder=5, label="Noisy Lab Data")
    ax.set_title(title)
    ax.set_ylim(-2.2, 2.2)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 🏋️ 练习 04a-2：对比不同核函数（Matérn vs RBF）的光滑度先验

RBF 核假设底层物理函数是**无穷阶连续可导（极其光滑）**的。但在相变边界或粗糙响应面上，物理量往往没有那么光滑。scikit-learn 提供了更灵活的 **Matérn 核（`Matern(length_scale=1.0, nu=...)`）**，通过参数 $\nu$ 控制可导阶数：
- `nu=0.5`：等价于 Ornstein–Uhlenbeck 过程（连续但不可导，呈折线毛刺状）；
- `nu=1.5`：一阶可导（工程与材料优化中最常使用的默认核）；
- `nu=2.5`：二阶可导；当 $\nu \to \infty$ 时收敛为 `RBF` 核。

请在上面的带噪数据 `(X_train, y_noisy)` 上，分别用 `Matern(1.0, nu=0.5)`、`Matern(1.0, nu=1.5)` 和 `RBF(1.0)`（均搭配 `C(1.0) * ... + WhiteKernel(0.1)`）拟合 GPR，并排画出三者的均值曲线与 95% 置信区间。

In [ ]:
# 在下方填写你的代码：


---

## 6. 离散分子空间的 GPR：Tanimoto 核函数与自带适用域预警

前面处理的自变量 $x$ 是连续工艺参数（如温度）。如果我们想对**离散分子结构**（用 2048 维 0/1 Morgan 指纹向量 $\mathbf{a}, \mathbf{b}$ 表示）做高斯过程回归并获得每个分子的预测不确定性，应该用什么核函数？

在第一讲 `01d` 和第三讲 `03b` 中我们已经知道，度量二值或计数分子指纹最自然的工具是 **Tanimoto 相似度（Jaccard 相似度）**：

$$K_{\text{Tanimoto}}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - \mathbf{a} \cdot \mathbf{b}}$$

在数学上可以严格证明：$K_{\text{Tanimoto}}$ 是一个满足 Mercer 定理的**正定核函数（Positive Semi-Definite Kernel）**！因此，我们无需任何复杂的转化，直接把分子间的 Tanimoto 相似度矩阵代入第 4 节的 GPR 解析公式，就能构建出**分子指纹高斯过程（Tanimoto-GPR）**。

下面我们从 `data/solubility.csv`（AqSolDB）中抽取 600 个分子（400 个作为训练集、200 个作为测试集），检验一个核心问题：**被 Tanimoto-GPR 标记为“高不确定性 $\sigma$”的测试集分子，它们的实际预测误差 $|\hat{y} - y|$ 是否真的更大？**

In [ ]:
df = pd.read_csv(find_file("solubility.csv"))
sol_col = "Solubility" if "Solubility" in df.columns else "logS"

sub = df.sample(700, random_state=42).reset_index(drop=True)
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fps, ys = [], []
for smi, logs in zip(sub["SMILES"], sub[sol_col]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fps.append(fp_gen.GetFingerprintAsNumPy(mol).astype(float))
        ys.append(logs)
    if len(fps) == 600:
        break

X_fp = np.array(fps)
y_logs = np.array(ys)
X_tr_fp, X_te_fp = X_fp[:400], X_fp[400:]
y_tr_fp, y_te_fp = y_logs[:400], y_logs[400:]

def tanimoto_kernel(A, B):
    dot = A @ B.T
    norm_a = np.sum(A**2, axis=1, keepdims=True)
    norm_b = np.sum(B**2, axis=1, keepdims=True).T
    denom = norm_a + norm_b - dot
    return np.where(denom > 0, dot / denom, 0.0)

# 用 Tanimoto 核矩阵计算 GPR 后验均值与标准差
noise_var = 0.15
K_tr = tanimoto_kernel(X_tr_fp, X_tr_fp) + noise_var * np.eye(len(X_tr_fp))
K_s  = tanimoto_kernel(X_tr_fp, X_te_fp)
K_ss_diag = np.ones(len(X_te_fp))   # 任意非空分子与自身的 Tanimoto 相似度恒为 1.0

y_mean = y_tr_fp.mean()
K_inv  = np.linalg.inv(K_tr)
mu_te  = y_mean + K_s.T @ K_inv @ (y_tr_fp - y_mean)
var_te = K_ss_diag - np.sum((K_s.T @ K_inv) * K_s.T, axis=1)
std_te = np.sqrt(np.maximum(var_te, 0))

abs_err = np.abs(mu_te - y_te_fp)
rmse = np.sqrt(np.mean((mu_te - y_te_fp)**2))
print(f"400 个分子训练的 Tanimoto-GPR 测试集 RMSE = {rmse:.2f}")

median_std = np.median(std_te)
low_unc_mae  = abs_err[std_te <= median_std].mean()
high_unc_mae = abs_err[std_te > median_std].mean()
print(f"模型最有把握的一半分子 (std <= {median_std:.2f}) 平均绝对误差 MAE = {low_unc_mae:.2f}")
print(f"模型最没把握的一半分子 (std >  {median_std:.2f}) 平均绝对误差 MAE = {high_unc_mae:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
sc = axes[0].scatter(y_te_fp, mu_te, c=std_te, cmap="viridis", alpha=0.8, s=28)
axes[0].plot([-10, 2], [-10, 2], "r--", lw=1)
axes[0].set_xlabel("Measured logS")
axes[0].set_ylabel("Predicted logS")
axes[0].set_title("Parity Plot Colored by GP Uncertainty (std)")
plt.colorbar(sc, ax=axes[0], label="Predicted std")

axes[1].scatter(std_te, abs_err, alpha=0.65, s=25)
axes[1].axvline(median_std, color="red", ls="--", label=f"Median std = {median_std:.2f}")
axes[1].set_xlabel("Predicted Uncertainty (std)")
axes[1].set_ylabel("Absolute Error |pred - true|")
axes[1].set_title("Higher Uncertainty Correlates with Larger Error")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

实验结果清晰证明了 Tanimoto-GPR 的不确定性校准能力：模型自身评估不确定性较低的前 50% 分子，实际平均绝对误差（MAE）显著低于不确定性较高的后 50% 分子。换句话说，**Tanimoto-GPR 把我们在 `02b` 和 `03a` 中手动编写的“最近邻相似度适用域检查”直接内化成了模型的原生输出 $\sigma(x)$**。

### 🏋️ 练习 04a-3：预测有机半导体分子的溶解度与不确定性 $\sigma$

利用第 6 节已经算好的 `X_tr_fp`、`K_inv` 和 `y_mean`，计算以下 5 个有机半导体相关分子的预测溶解度 `Pred logS`、预测标准差 `Pred std`，以及它们在 400 个训练分子中的最大 Tanimoto 相似度 `Max Tanimoto Sim`：
- `Benzene`: `"c1ccccc1"`
- `Thiophene`: `"c1ccsc1"`
- `Anthracene`: `"c1ccc2cc3ccccc3cc2c1"`
- `Tetracene`: `"c1ccc2cc3cc4ccccc4cc3cc2c1"`
- `C8-BTBT`: `"CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc21"`

将结果整理成一张 DataFrame 表格，观察 `Max Tanimoto Sim` 与 `Pred std` 之间严格的反比关系。

In [ ]:
# 在下方填写你的代码：


---

## 小结

| 核心概念 | 数学本质与工程实现 | 材料与化学物理意义 |
|---|---|---|
| **高斯过程先验（GP Prior）** | 任意有限点集服从联合正态分布 $\mathcal{N}(\mathbf{0}, K)$ | 在未观测实验数据前，由核函数定义什么样的响应曲线是光滑合理的 |
| **核函数（RBF / Matérn / White）** | `C() * RBF(l) + WhiteKernel()` | 长度尺度 $\ell$ 控制性质随工艺参数变化的快慢；`WhiteKernel` 自动剥离实验测量误差 |
| **高斯过程后验（GP Posterior）** | $\boldsymbol{\mu}_* = K_*^T (K + \sigma_n^2 I)^{-1} \mathbf{y}$<br>$\Sigma_* = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$ | 同时给出预测均值与预测不确定性 $\sigma(x)$；远离已测点时不确定性自动膨胀报警 |
| **Tanimoto 分子核** | $K(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|^2+\|\mathbf{b}\|^2-\mathbf{a}\cdot\mathbf{b}}$ | 无需连续坐标即可在分子指纹空间做 GPR，将化学适用域（AD）直接量化为预测标准差 |

**下一节预告（`04b_bayesian_optimization.ipynb`）**：既然高斯过程已经能同时告诉我们“哪里预测性能高（$\mu$ 大）”和“哪里还没探索过（$\sigma$ 大）”，我们该如何把这两者结合成一个自动推荐指标，用最少的实验次数找到最优的材料工艺或合成反应配方？